<a href="https://colab.research.google.com/github/SEC-API-io/sec-api-cookbook/blob/main/notebooks/registration-statements/registration-statements-prospectus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Registration Statements & Prospectus Filings with Python


## Quick Start

In [ ]:
!pip install sec-api

In [ ]:
from sec_api import Form_S1_424B4_Api

form_s1_424B4_api = Form_S1_424B4_Api("YOUR_API_KEY")

query = {
    "query": "ticker:V",
    "from": "0",
    "size": "50",
    "sort": [{"filedAt": {"order": "desc"}}],
}

response = form_s1_424B4_api.get_data(query)

In [ ]:
import json
print(json.dumps(response["data"][0], indent=2))

{
  "id": "b4f377b5fcd1c9ea391ba9e8c6c5ab11",
  "filedAt": "2008-03-19T17:14:29-04:00",
  "accessionNo": "0001193125-08-060989",
  "formType": "424B4",
  "cik": "1403161",
  "ticker": "V",
  "entityName": "Visa Inc.",
  "filingUrl": "https://www.sec.gov/Archives/edgar/data/1403161/000119312508060989/d424b4.htm",
  "tickers": [
    {
      "ticker": "V",
      "type": "Class A Common Stock",
      "exchange": "New York Stock Exchange"
    }
  ],
  "securities": [
    {
      "name": "406,000,000 Shares Class A Common Stock"
    }
  ],
  "publicOfferingPrice": {
    "perShare": 44,
    "perShareText": "$44.000",
    "total": 17864000000,
    "totalText": "$17,864,000,000"
  },
  "underwritingDiscount": {
    "perShare": 1.232,
    "perShareText": "$1.232",
    "total": 500192000,
    "totalText": "$500,192,000"
  },
  "proceedsBeforeExpenses": {
    "perShare": 42.768,
    "perShareText": "$42.768",
    "total": 17363808000,
    "totalText": "$17,363,808,000"
  },
  "underwriters": [
   

## Loading Large Amounts of Data

Many use cases require loading large amounts of structured data from the Registration Statement & Prospectus Data API, for example loading all data for 424B4 prospectuses from 2010 to 2023. The following code demonstrates how to achieve this by iterating over the range of years from `start_year` to `end_year` and fetching all 424B4 prospectuses for each month of each year.

The query can be adjusted and tailored to your specific needs. For example, if you are interested in S-1 registration statements, you would simply change the value of the `formType` parameter to `"S-1"`. Note, the quotation marks around `"S-1"` are required otherwise your search query also matches on `F-1` and `S-11` forms. The updated `query_string` would look like this:

`f'filedAt:[{start_date} TO {end_date}] AND formType:"S-1"'`

In case you are interested in all registration statements and prospectuses, you can simply remove the `formType` parameter from the `query_string`:

`f'filedAt:[{start_date} TO {end_date}]'`

In [ ]:
def load_data_from_api(start_year=2023, end_year=2024):
    years = list(range(start_year, end_year + 1))
    months = [f"{month:02d}" for month in range(1, 13)]  # 01, 02, ..., 12
    data = []

    for year in years:
        year_counter = 0

        for month in months:
            has_more = True
            query_from = 0
            query_size = 50
            month_counter = 0

            while has_more:
                start_date = f"{year}-{month}-01"
                end_date = f"{year}-{month}-31"
                query_string = (
                    f'filedAt:[{start_date} TO {end_date}] AND formType:424B4'
                )
                query = {
                    "query": query_string,
                    "from": query_from,
                    "size": query_size,
                    "sort": [{"filedAt": {"order": "desc"}}],
                }

                response = form_s1_424B4_api.get_data(query)
                if len(response["data"]) == 0:
                    break
                data += response["data"]
                query_from += query_size
                month_counter += len(response["data"])

            year_counter += month_counter
            print(f"✅ {year}-{month} with {month_counter} filings")

        print(f"✅ {year} with {year_counter} filings")

    return data

In [ ]:
forms_424B4 = load_data_from_api(start_year = 2023, end_year = 2023)

✅ 2023-01 with 22 filings
✅ 2023-02 with 39 filings
✅ 2023-03 with 34 filings
✅ 2023-04 with 32 filings
✅ 2023-05 with 33 filings
✅ 2023-06 with 48 filings
✅ 2023-07 with 30 filings
✅ 2023-08 with 36 filings
✅ 2023-09 with 34 filings
✅ 2023-10 with 35 filings
✅ 2023-11 with 40 filings
✅ 2023-12 with 27 filings
✅ 2023 with 410 filings


In [ ]:
forms_424B4[:2]

[{'id': '507481881b3e4416d56763903035a635',
  'filedAt': '2023-01-31T14:46:26-05:00',
  'accessionNo': '0001493152-23-003123',
  'formType': '424B4',
  'cik': '1879726',
  'ticker': 'SIDU',
  'entityName': 'Sidus Space Inc.',
  'filingUrl': 'https://www.sec.gov/Archives/edgar/data/1879726/000149315223003123/form424b4.htm',
  'tickers': [{'ticker': 'SIDU',
    'type': 'Class A Common Stock',
    'exchange': 'The Nasdaq Capital Market'}],
  'securities': [{'name': '2,640,000 shares of Class A Common Stock'},
   {'name': 'Pre-Funded Warrants to Purchase 12,360,000 shares of Class A Common Stock'},
   {'name': 'Class A common stock'},
   {'name': 'Class B common stock'}],
  'publicOfferingPrice': {'perShare': 0.3,
   'perShareText': '$0.30',
   'total': 4500000,
   'totalText': '$4,500,000'},
  'underwritingDiscount': {'perShare': 0.021,
   'perShareText': '$0.021',
   'total': 315000,
   'totalText': '$315,000'},
  'proceedsBeforeExpenses': {'perShare': 0.279,
   'perShareText': '$0.279',